# Twój generator 5 mln: analiza pliku partiami

Źródła: AI_analiza_danych_tools_11/DZIEN_1 (generator i analiza) oraz motyw agregacji z SANTANDER_ANALIZA_BIG_DATA/DZIEN_1/dask_python.ipynb. 35 minut. Standardowo 100 tys. pozycji i pandas bez Dask/Spark. Liczba wierszy sama nie definiuje Big Data. Ćwiczymy kontrolę zużycia pamięci i poprawność agregacji.

Uruchom komórkę Start. Cały pakiet musi być rozpakowany. W Colab prześlij ZIP i rozpakuj go przez panel Pliki lub komendę `!unzip nazwa.zip`, a następnie ustaw katalog roboczy na folder pakietu.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'dane').exists() and (ROOT.parent/'dane').exists(): ROOT=ROOT.parent
if not (ROOT/'dane').exists(): raise FileNotFoundError('Rozpakuj cały pakiet i otwórz notebook w folderze zawierającym dane oraz narzedzia.')
sys.path.insert(0,str(ROOT/'narzedzia'))
from swps_tools import sales_quality, aggregate_chunks, to_cents, AmountCapper, make_transactions

## 1. Format i próbka

Wczytaj pierwsze 5000 wierszy elektronika_100k.csv z właściwym separatorem, kodowaniem i przecinkiem dziesiętnym. Ustal jednostkę oraz wartość pozycji. Zapisz, które wielkości próbka opisuje dokładnie, a które tylko przybliża.

In [ ]:
path=ROOT/'dane/elektronika_100k.csv'
sample=pd.read_csv(path,sep=';',encoding='utf-8-sig',decimal=',',nrows=5000)
print(sample.head().to_string(index=False));print(sample.dtypes)
assert np.allclose(sample['wartość netto'],sample['ilość sztuk']*sample['cena netto'])
print('Średnia ceny w PRÓBCE:',sample['cena netto'].mean())

## 2. Agregacja całego pliku

Przeczytaj implementację aggregate_chunks w narzedzia/swps_tools.py. Policz cały plik w partiach 17003 rekordy. Zauważ, że ostatnia partia jest mniejsza. Podaj n, sumę sztuk, sumę wartości w groszach i średnią ceny jednostkowej. Nie konkatenować wszystkich partii.

In [ ]:
result=aggregate_chunks(path,chunksize=17003)
print({k:v for k,v in result.items() if k!='groups'})
print(result['groups'].to_string())
assert result['rows']==100000
assert result['groups'].n.sum()==result['rows']
assert result['groups'].net_cents.sum()==result['net_cents']

## 3. Średnia partii

Policz średnie cen w partiach po 17003, a potem ich zwykłą średnią. Porównaj z sumą wszystkich cen dzieloną przez liczbę pozycji. Skąd może brać się różnica? Porównaj także średnią cen pozycji z ceną ważoną ilością sztuk.

In [ ]:
means=[];counts=[];sums=[]
for part in pd.read_csv(path,sep=';',encoding='utf-8-sig',usecols=['cena netto'],dtype=str,chunksize=17003):
    cents=to_cents(part['cena netto'])
    means.append(cents.mean());counts.append(len(part));sums.append(int(cents.sum()))
wrong=np.mean(means)/100
correct=sum(sums)/sum(counts)/100
print('Rozmiary partii:',counts)
print('Średnia średnich:',wrong,'poprawna średnia pozycji:',correct)
print('Cena ważona sztukami:',result['quantity_weighted_price_pln'])
assert np.isclose(correct,result['mean_unit_price_pln'])
assert np.isclose(np.average(means,weights=counts)/100,correct)

## 4. Niezależna kontrola

Dla dostarczonych 100 tys. pozycji możesz jednorazowo wczytać cały plik i potwierdzić sumy. To kontrola dydaktyczna na małym wariancie. Główna metoda z partii działa bez pełnego wczytania. Dla 5 mln użyj generatora i agregacji, nie włączaj poniższej kontroli bez oceny RAM.

In [ ]:
reference=pd.read_csv(path,sep=';',encoding='utf-8-sig',dtype=str)
ref_net=to_cents(reference['wartość netto'])
ref_price=to_cents(reference['cena netto'])
assert result['net_cents']==int(ref_net.sum())
assert result['price_cents_sum']==int(ref_price.sum())
assert result['units']==int(reference['ilość sztuk'].astype('int64').sum())
print('Kontrola sum pełnego odczytu i partii: OK')
del reference,ref_net,ref_price

## Wariant 5 mln
W terminalu folderu pakietu: `python narzedzia/generuj_dane_swps.py --n 5000000 --output dane/elektronika_5mln.csv`. Potem `aggregate_chunks(ROOT / "dane/elektronika_5mln.csv", chunksize=100000)`. Oczekuj setek MB na dysku. Generator zapisuje partie; analiza utrzymuje pojedynczą partię i niewielki stan grup. Nie obiecujemy, że sam chunksize ogranicza pamięć wszystkich możliwych algorytmów. Dokładne globalne kwantyle i pełna walidacja są w oryginalnym analizatorze dołączonym w narzedzia.